# 02 — Sélection des modèles et justification des hyperparamètres

**Projet :** Dog Breed Classifier — MLOps  
**Auteurs :** Équipe projet  
**Objectif :** Ce notebook est **purement documentaire** : aucun entraînement n'est réalisé ici.  
Il justifie, par la littérature scientifique et les documentations officielles, les choix d'architectures
et les plages d'hyperparamètres qui seront explorées par Optuna lors de la phase d'entraînement.

---

## Contexte du problème

**Tâche :** Classification fine-grained (FGVC) de 10 races de chiens  
**Dataset :** Sous-ensemble du Stanford Dogs Dataset — ~1 635 images, split 70/15/15  
**Défi :** Grande variance intra-classe (poses, éclairages, angles variables pour une même race)
et faible variance inter-classe (races visuellement proches).

> *"Fine-grained visual recognition [...] involves identifying subordinate categories within a broader category,
such as bird species or dog breeds. These problems are challenging due to large intra-class variation and small
inter-class variation."*  
> — **Zhuang et al., IEEE 2021** · [lien](https://ieeexplore.ieee.org/document/9295723/)


---
## 1. Pourquoi le transfer learning sur un petit dataset ?

Sur ~1 144 images d'entraînement (split 70% de 1 635), un CNN entraîné de zéro atteint des performances
significativement inférieures à celles du transfer learning — la littérature quantifie un écart de
10 à 30 points d'accuracy sur des datasets de taille comparable.

| Approche | Accuracy attendue — 120 races (≈) |
|---|---|
| CNN scratch (baseline) | 60 – 70 % |
| Transfer learning EfficientNetB0 | 85 – 95 %+ |
| Transfer learning InceptionV3 | 85 – 95 %+ |


arXiv 2601.02246 montre que le transfer learning surpasse l'entraînement de zéro de 10 à 30 points
d'accuracy sur des petits datasets d'images.  
— **arXiv 2601.02246** · *A Comparative Study of Custom CNNs vs. Pre-trained Models* · [lien](https://arxiv.org/html/2601.02246v1)

Les modèles pré-entraînés sur ImageNet (1,28 M images, 1 000 classes) ont appris des représentations
hiérarchiques (textures, formes, parties d'objets) directement réutilisables pour distinguer des races de chiens.

---
## 2. Comparaison des architectures candidates

### 2.1 Tableau de synthèse

| Modèle | Top-1 ImageNet | Paramètres | FLOPs | Décision |
|---|---|---|---|---|
| **EfficientNetB0** | 77,1 % | 5,3 M | 0,39 B | **Retenu** |
| **InceptionV3** | 78,8 % | 23,8 M | 5,7 B | **Retenu** |
| ResNet-50 | 76,0 % | 26 M | 4,1 B | Écarté |
| VGG-16 | 71,3 % | 138 M | 15,5 B | Écarté |
| MobileNetV2 | 72,0 % | 3,2 M | 0,3 B | Écarté |

Sources : **Tan & Le, ICML 2019** pour EfficientNet · [arXiv:1905.11946](https://arxiv.org/pdf/1905.11946) — **Szegedy et al., CVPR 2016** pour InceptionV3 · [arXiv:1512.00567](https://arxiv.org/abs/1512.00567) — chiffres VGG/ResNet/MobileNet : papiers originaux respectifs.

In [1]:
import tensorflow as tf

# Instancier les deux modèles retenus et afficher leur nombre de paramètres
# (weights="imagenet" confirme qu'ils sont disponibles pré-entraînés)
efficient_net = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet")
inception     = tf.keras.applications.InceptionV3(include_top=False, weights="imagenet")

print(f"EfficientNetB0 : {efficient_net.count_params():>10,} paramètres")
print(f"InceptionV3    : {inception.count_params():>10,} paramètres")
print(f"Ratio           : InceptionV3 est {inception.count_params()/efficient_net.count_params():.1f}× plus lourd")


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 26s 0us/step
EfficientNetB0 :  4,049,571 paramètres
InceptionV3    : 21,802,784 paramètres
Ratio           : InceptionV3 est 5.4× plus lourd


### 2.2 Justification — EfficientNetB0 

- **Meilleur ratio accuracy / coût computationnel** parmi toutes les architectures testées.  
  > *"EfficientNets achieve both better accuracy and better efficiency over existing CNNs."*  
  > — **Tan & Le, ICML 2019** · *EfficientNet: Rethinking Model Scaling for CNNs* · [arXiv:1905.11946](https://arxiv.org/pdf/1905.11946)

- 5,3 M paramètres : assez expressif pour capturer les détails fins, assez compact pour éviter
  l'overfitting sur 1 635 images.

### 2.3 Justification — InceptionV3 

- Architecture **multi-échelle** (convolutions 1×1, 3×3, 5×5 en parallèle dans chaque module Inception) :
  idéale pour capturer simultanément les textures fines (poils, oreilles) et les formes globales (silhouette).

- **91,2 % d'accuracy sur classification de races** (120 races, Stanford Dogs complet) —
  meilleure performance parmi les modèles testés dans l'étude.  
  > — **ASPD / IJES 2023** · *Automated Dog Breed Recognition: A Transfer Learning Study* · [lien](https://theaspd.com/index.php/ijes/article/download/6503/4701/13220)

- Sur notre sous-problème à 10 races, les performances seront supérieures.

### 2.4 Architectures écartées — justifications

| Architecture | Raison d'exclusion |
|---|---|
| **VGG-16/19** | 138 M paramètres → overfitting probable sur 1 635 images ; architecture sans skip connections, dépassée |
| **ResNet-50** | 5× plus lourd qu'EfficientNetB0 (26 M vs 5,3 M) pour un gain marginal sur ce dataset |
| **MobileNetV2/V3** | Conçu pour l'inférence embarquée (mobile) ; accuracy FGVC inférieure à EfficientNetB0 |
| **EfficientNetB3** | Plafond d'accuracy légèrement supérieur (12 M params) mais risque d'overfitting sur 1 635 images |
| **ViT / RAMS-Trans** | État de l'art (91,4 % FGVC) mais requiert des données massives |

---
## 3. CNN scratch — baseline indispensable

Un CNN entraîné de zéro sert de baseline. La littérature prévoit +10 à +30 points d'accuracy pour le transfer
learning sur des petits datasets.

Architecture utilisée : 3 blocs Conv→Pool + GlobalAveragePooling + Dense head.


In [2]:
import tensorflow as tf

def build_cnn_scratch(num_classes: int = 10, input_shape: tuple = (224, 224, 3)):
    """
    CNN entraîné de zéro — sert de baseline.
    Architecture volontairement simple pour éviter l'overfitting sur ~1 635 images.
    """
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=input_shape),

        # Bloc 1 — détection bas niveau (contours, textures)
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),

        # Bloc 2 — features intermédiaires
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),

        # Bloc 3 — features haut niveau (parties spécifiques à la race)
        tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),

        # Réduction spatiale → vecteur de features
        tf.keras.layers.GlobalAveragePooling2D(),

        # Classification head
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation="softmax"),
    ], name="cnn_scratch_baseline")
    return model

cnn = build_cnn_scratch()
cnn.summary()
print(f"\nTotal paramètres entraînables : {cnn.count_params():,}")


Model: "cnn_scratch_baseline"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_94 (Conv2D)                   │ (None, 224, 224, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 112, 112, 32)        │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_95 (Conv2D)                   │ (None, 112, 112, 64)        │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 56, 56, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_96 (Conv2D)                   │ (None, 56, 56, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_6 (MaxPooling2D)       │ (None, 28, 28, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 128)                 │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 128,842 (503.29 KB)

 Trainable params: 128,842 (503.29 KB)

 Non-trainable params: 0 (0.00 B)


Total paramètres entraînables : 128,842


---
## 4. Stratégie de fine-tuning en 2 phases

| Phase | Couches entraînées | Epochs | Learning rate |
|---|---|---|---|
| **Phase 1** — warm-up | Head uniquement (base gelée) | ~25 | `1e-4` |
| **Phase 2** — fine-tuning | Head + top 20 blocs (BatchNorm gelées) | 10 – 30 + early stopping | `1e-5` |

La base est gelée en phase 1 pour éviter le *catastrophic forgetting* — la tête étant initialisée aléatoirement, ses gradients détruiraient les poids ImageNet dès les premières itérations.  
En phase 2, un LR très faible (`1e-5`) permet d'affiner les dernières couches sans effacer les représentations pré-entraînées.

Sources : **Keras Transfer Learning Guide** · [lien](https://keras.io/guides/transfer_learning/) — **Keras EfficientNet Tutorial** · [lien](https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/)

In [1]:
import tensorflow as tf

# ── EfficientNetB0 : inspection des couches avant/après dégel ──
base_eff = tf.keras.applications.EfficientNetB0(include_top=False, weights="imagenet")

print("=" * 60)
print("EFFICIENTNETB0")
print("=" * 60)
print(f"  Nombre total de couches : {len(base_eff.layers)}")

# Par défaut, toutes les couches sont trainable=True à l'instanciation.
# En pratique, on gèle la base avec base.trainable = False avant de construire le modèle complet.
print(f"  Toutes couches trainable=True par défaut : {all(l.trainable for l in base_eff.layers)}")

print("\n  Couches dégelées en phase 2 (top 20, BatchNorm restent gelées) :")
count_unfrozen = 0
for layer in base_eff.layers[-20:]:
    if not isinstance(layer, tf.keras.layers.BatchNormalization):
        print(f"    ✓ {layer.name}")
        count_unfrozen += 1
    else:
        print(f"    ✗ {layer.name}  ← BatchNorm reste gelée")
print(f"  Total dégelées en phase 2 : {count_unfrozen} couches")
print()
print("  Note : le gel effectif se fait dans train.py via base.trainable = False")
print("  Source : Keras EfficientNet Tutorial")
print("  https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/")

EFFICIENTNETB0
  Nombre total de couches : 238
  Toutes couches trainable=True par défaut : True

  Couches dégelées en phase 2 (top 20, BatchNorm restent gelées) :
    ✓ block6d_project_conv
    ✗ block6d_project_bn  ← BatchNorm reste gelée
    ✓ block6d_drop
    ✓ block6d_add
    ✓ block7a_expand_conv
    ✗ block7a_expand_bn  ← BatchNorm reste gelée
    ✓ block7a_expand_activation
    ✓ block7a_dwconv
    ✗ block7a_bn  ← BatchNorm reste gelée
    ✓ block7a_activation
    ✓ block7a_se_squeeze
    ✓ block7a_se_reshape
    ✓ block7a_se_reduce
    ✓ block7a_se_expand
    ✓ block7a_se_excite
    ✓ block7a_project_conv
    ✗ block7a_project_bn  ← BatchNorm reste gelée
    ✓ top_conv
    ✗ top_bn  ← BatchNorm reste gelée
    ✓ top_activation
  Total dégelées en phase 2 : 15 couches

  Note : le gel effectif se fait dans train.py via base.trainable = False
  Source : Keras EfficientNet Tutorial
  https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/


**Pourquoi les BatchNormalization restent-elles toujours gelées ?**

Les BatchNorm mémorisent les statistiques de distribution (moyenne et variance) calculées sur
ImageNet (1,2 M images). Si on les dégèle sur 1 144 images d'entraînement (notre train set),
ces statistiques seraient recalculées sur un échantillon statistiquement insuffisant — ce qui
détruirait les représentations pré-entraînées.

> Source : **Keras EfficientNet Tutorial** · [lien](https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/)


In [2]:
# ── InceptionV3 : identifier les blocs à dégeler ──
base_iv3 = tf.keras.applications.InceptionV3(include_top=False, weights="imagenet")

print("=" * 60)
print("INCEPTIONV3")
print("=" * 60)
print(f"  Nombre total de couches : {len(base_iv3.layers)}")
print(f"  Layer 249 (mixed7 debut) : {base_iv3.layers[249].name}")
print(f"  Layer 280 (mixed8 debut) : {base_iv3.layers[280].name}")
print()
print("  Stratégie recommandée pour petit dataset :")
print("  → Dégeler à partir de mixed8 (layer ~280) : plus conservateur")
print("  → Dégeler à partir de mixed7 (layer 249)  : si overfitting faible")
print()
print("  Source : Keras GitHub issue #10554")
print("  https://github.com/keras-team/keras/issues/10554")


INCEPTIONV3
  Nombre total de couches : 311
  Layer 249 (mixed7 debut) : conv2d_80
  Layer 280 (mixed8 debut) : conv2d_89

  Stratégie recommandée pour petit dataset :
  → Dégeler à partir de mixed8 (layer ~280) : plus conservateur
  → Dégeler à partir de mixed7 (layer 249)  : si overfitting faible

  Source : Keras GitHub issue #10554
  https://github.com/keras-team/keras/issues/10554


---
## 5. Hyperparamètres et plages de recherche


### Tableau de synthèse

| Hyperparamètre | Plage Optuna | Source principale |
|---|---|---|
| Learning rate (phase 2) | `1e-5` à `1e-4` (log) | Keras Transfer Learning Guide |
| Dropout (head) | `0.2` à `0.4` | Keras EfficientNet Tutorial |
| Batch size | 16 ou 32 | Keskar et al. ICLR 2017 + Masters & Luschi 2018 |
| Couches dégelées EfficientNetB0 | Top 20 (sans BatchNorm) | Keras EfficientNet Tutorial |
| Couches dégelées InceptionV3 | Layer 249 ou 280 | Keras GitHub #10554 |
| L2 regularization | `1e-4` à `1e-3` (log) | Li et al. ICLR 2020 |


---
### 5.1 Learning rate — phase 2

**Plage :** `1e-5` à `1e-4` (échelle logarithmique)

Les deux sources officielles Keras convergent sur `1e-5` comme valeur de référence pour la phase 2 :

> *"use a very low learning rate at this stage, because you are training a much larger model than in
the first round of training, on a dataset that is typically very small."* → `Adam(learning_rate=1e-5)`  
> — **Keras Transfer Learning Guide** · [lien](https://keras.io/guides/transfer_learning/)

> `optimizer = keras.optimizers.Adam(learning_rate=1e-5)` dans `unfreeze_model()`  
> — **Keras EfficientNet Tutorial** · [lien](https://keras.io/examples/vision/image_classification_efficientnet_fine_tuning/)

**Pourquoi pas `1e-3` :**  
Un LR ≥ 1e-3 provoque le *catastrophic forgetting* — les grands pas de gradient écrasent
les poids pré-entraînés sur ImageNet en quelques itérations.

**Pourquoi pas `1e-6` :**  
Convergence quasi nulle, adaptation au domaine cible (chiens) inexistante — équivalent à
garder la base gelée.

**Pourquoi Optuna plutôt qu'une valeur fixe :**  
Li et al. (ICLR 2020) montrent que la plage optimale dépend de la similarité entre le domaine
source (ImageNet) et le domaine cible — ici modérément similaire. Une recherche automatisée
est justifiée.  
> — **Li et al., ICLR 2020** · arXiv:2002.11770 · [lien](https://arxiv.org/abs/2002.11770)


---
### 5.2 Dropout (classification head)

**Plage :** `0.2` à `0.4`  
**Valeur de référence :** `0.2`

Trois sources officielles convergent sur `0.2` :

| Source | Valeur |
|---|---|
| Keras EfficientNet Tutorial | `top_dropout_rate = 0.2` (note : peut monter à `0.4`) |
| Keras Transfer Learning Guide | `keras.layers.Dropout(0.2)(x)` |
| TensorFlow Transfer Learning Tutorial | `tf.keras.layers.Dropout(0.2)(x)` |

**Pourquoi pas `0.5` (valeur du papier original Dropout) :**

> *"Dropout of 0.5 is optimal for fully connected layers with many parameters."*  
> — **Srivastava et al., JMLR 2014** · *Dropout: A Simple Way to Prevent Neural Networks from
Overfitting* · [lien](https://jmlr.org/papers/v15/srivastava14a.html)

En transfer learning, la **base gelée agit déjà comme un régulariseur implicite fort** — seul le
head (quelques milliers de paramètres) est entraîné en phase 1. Un dropout de `0.5` sur un head
aussi petit risquerait le sous-apprentissage.


---
### 5.3 Batch size

**Valeurs explorées :** 16 ou 32

**Keskar et al., ICLR 2017 (arXiv:1609.04836) :**  
> *"Large-batch methods tend to converge to sharp minimizers [...] sharp minima lead to poorer
generalization."*  
Les petits batchs introduisent du bruit stochastique qui agit comme régulariseur implicite et
guide l'optimisation vers des minima plats qui généralisent mieux.  
[lien](https://arxiv.org/abs/1609.04836)

**Masters & Luschi, 2018 (arXiv:1804.07612) :**  
> *"The best performance has been consistently obtained for mini-batch sizes between m=2 and m=32."*  
Citent directement la borne supérieure de 32 comme valeur empiriquement validée.  
[lien](https://arxiv.org/abs/1804.07612)

Ces deux résultats convergent vers batch=16 ou 32 comme plage optimale pour un dataset de ~1 600 images.

> Note : le Keras EfficientNet Tutorial utilise batch=64, mais sur ~10 000 images (Stanford Dogs
complet). Sur notre sous-ensemble de 1 144 images train, les deux papiers ci-dessus justifient
de rester à 32 ou en-dessous.

---
### 5.4 L2 regularization (weight decay)

**Plage :** `1e-4` à `1e-3` (échelle logarithmique)

- **Li et al., ICLR 2020** utilisent `wd=1e-4` comme baseline dans leurs expériences de fine-tuning.  
  [arXiv:2002.11770](https://arxiv.org/abs/2002.11770)

- **Analyse fine-tuning ACM 2019** : grid search sur `{1e-3, 1e-4, 1e-5, 1e-6, 0}` →
  `1e-4` à `5e-4` performent systématiquement mieux.  
  [ACM DL](https://dl.acm.org/doi/fullHtml/10.1145/3319500)

**Bornes :**
- En-dessous de `1e-5` → trop faible pour régulariser sur 1 635 images.
- Au-dessus de `1e-3` → écrase les poids pré-entraînés, annule le bénéfice du transfer learning.

**Note :** Li et al. (ICML 2018, arXiv:1802.01483) montrent que la L2 standard (pénalité vers zéro)
est sous-optimale pour le fine-tuning — il vaudrait mieux pénaliser l'écart aux poids pré-entraînés
(L2-SP). On utilise la L2 standard pour sa disponibilité native dans Keras, en ayant conscience
de cette limite.


---
### 5.5 Search space Optuna — synthèse

Le code ci-dessous définit l'espace de recherche qui sera utilisé dans `train.py`.
Aucun entraînement n'est lancé ici.


In [3]:
import optuna

def define_search_space(trial: optuna.Trial) -> dict:
    """
    Définit l'espace de recherche Optuna pour les hyperparamètres de fine-tuning.

    Justifications :
    - learning_rate : 1e-5 à 1e-4 (Keras TL Guide + EfficientNet Tutorial)
    - dropout_rate  : 0.2 à 0.4 (Keras tutorials + Srivastava et al. JMLR 2014)
    - batch_size    : 16 ou 32 (Keskar ICLR 2017 + Masters & Luschi 2018)
    - l2_reg        : 1e-4 à 1e-3 (Li et al. ICLR 2020 + ACM 2019)
    """
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True),
        "dropout_rate":  trial.suggest_float("dropout_rate",  0.2,  0.4),
        "batch_size":    trial.suggest_categorical("batch_size", [16, 32]),
        "l2_reg":        trial.suggest_float("l2_reg", 1e-4, 1e-3, log=True),
    }

# Afficher un exemple de tirage aléatoire
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
trial_example = study.ask()
params = define_search_space(trial_example)

print("Exemple de tirage Optuna (TPE Sampler) :")
for k, v in params.items():
    print(f"  {k:<20} : {v}")


C:\Users\Junior\AppData\Local\pypoetry\Cache\virtualenvs\dog-breed-classifier-4cMJHnNA-py3.12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-06-10 00:19:41,705] A new study created in memory with name: no-name-600d63a1-a807-4ed4-acc8-32e04a01b064


Exemple de tirage Optuna (TPE Sampler) :
  learning_rate        : 2.368863950364079e-05
  dropout_rate         : 0.3901428612819833
  batch_size           : 16
  l2_reg               : 0.0001432249371823026


---
## 6. Data augmentation

### Justification

Avec ~1 144 images d'entraînement, l'augmentation est **indispensable** pour régulariser
et améliorer la généralisation. Elle doit être appliquée **uniquement sur le split train**
(jamais sur val/test — sinon fuite de données).

**Consensus issu de la littérature FGVC :**

> Flip horizontal, rotation ±15–20°, zoom 0.8–1.0, décalage ±10–15%, variation luminosité ±20%  
> — **PMC / Bioengineering 2024** · *Classification of Dog Breeds Using CNN Models and SVM* · 
[lien](https://pmc.ncbi.nlm.nih.gov/articles/PMC11591900/)

**Transformations à éviter :**
- Flip vertical → non naturel pour des photos de chiens
- Crop agressif → risque de supprimer le chien du cadre
- Cisaillement fort → dégrade les features discriminantes (forme des oreilles, etc.)


In [ ]:
import tensorflow as tf

# Pipeline d'augmentation — appliqué UNIQUEMENT sur le split train dans train.py
# (jamais sur val/test pour éviter toute fuite de données)
augmentation_pipeline = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),          # p=0.5 — invariant à la race
    tf.keras.layers.RandomRotation(0.15),              # ±15° — modéré
    tf.keras.layers.RandomZoom(0.2),                   # zoom 0.8–1.2×
    tf.keras.layers.RandomTranslation(0.1, 0.1),       # décalage ±10%
    tf.keras.layers.RandomBrightness(0.2),             # luminosité ±20%
], name="data_augmentation_train_only")

augmentation_pipeline.summary()

print("\nCe pipeline sera intégré dans train.py, appliqué en ligne pendant l'entraînement.")
print("Les splits val et test utilisent uniquement resize + normalisation.")

---
## 7. Benchmarks de référence (Stanford Dogs)

Les chiffres ci-dessous concernent le dataset complet (120 races).
Sur notre sous-problème à **10 races**, les performances seront supérieures.

| Méthode | Backbone | Accuracy | Source |
|---|---|---|---|
| CNN scratch | Custom | ~60 – 70 % | Estimations multi-études |
| Transfer learning | InceptionV3 | **91,2 %** | ASPD / IJES 2023 |
| CNN + SVM | Multiple CNNs | **95,24 %** | PMC / Bioengineering 2024 |
| ViT-based | RAMS-Trans | 91,4 % | arXiv:2107.08192 |

**Références :**
- ASPD 2023 : [lien](https://theaspd.com/index.php/ijes/article/download/6503/4701/13220)
- PMC 2024 : [lien](https://pmc.ncbi.nlm.nih.gov/articles/PMC11591900/)
- Khosla et al., Stanford Dogs Dataset, CVPR 2011 : [lien](http://vision.stanford.edu/aditya86/ImageNetDogs/main.html)


---
## 8. Synthèse des décisions

| Décision | Choix | Justification |
|---|---|---|
| Modèles retenus | EfficientNetB0 + InceptionV3 + CNN scratch | Ratio perf/params + résultats FGVC + baseline |
| Stratégie fine-tuning | 2 phases (warm-up puis dégel progressif) | Keras Transfer Learning Guide + EfficientNet Tutorial |
| BatchNorm toujours gelée | Oui | Keras EfficientNet Tutorial |
| Couches dégelées EffNet | Top 20 (sans BN) | Keras EfficientNet Tutorial |
| Couches dégelées IV3 | À partir de layer 249 ou 280 | Keras GitHub #10554 |
| LR phase 2 | `1e-5` à `1e-4` | Keras TL Guide + Li et al. ICLR 2020 |
| Dropout head | `0.2` à `0.4` | Keras Tutorials + Srivastava JMLR 2014 |
| Batch size | 16 ou 32 | Keskar ICLR 2017 + Masters & Luschi 2018 |
| L2 reg | `1e-4` à `1e-3` | Li et al. ICLR 2020 + ACM 2019 |
| Augmentation | Flip H + Rotation + Zoom + Shift + Brightness | PMC Bioengineering 2024 |
| Optimiseur | Adam | Standard pour le fine-tuning TF/Keras |
| Recherche HPs | Optuna (TPE Sampler) | Explorations justifiées par la littérature ci-dessus |

---

*Ce notebook est un document de référence du projet. Les choix ici documentés sont implémentés dans
`dog_breed_classifier/modeling/train.py` et paramétrés via `params.yaml`.*